# Build the historical bilateral trade seed matrix

**Replaces** the CEPII-distance + country-fixed-effects approach in `5-1-trade-gravity-matrix.ipynb`.

We use observed bilateral trade matrices (UN Comtrade-style, 142 × 142, 1962–2022) directly as the **seed** for a Stone (1962) RAS / IPF procedure in `5-2`. This preserves the structural geography of real aggregate trade — UAE–Gulf corridors, Norway–Northern Europe, Malaysia–Singapore, China–Hong Kong, Canada–USA, etc. — instead of relying on a generic distance-decay kernel that misses these patterns.

## Pipeline

```
(F:) ...trade-142/{cat}-matrix-{year}.csv   ←  bilateral matrices, unit = kg
          |
          | average over 2018–2022
          | sum across {other-sand, gravel-crushed-stone, limestone}
          | convert kg → Mt
          | add tiny ε floor for novel pairs
          v
results/historical-trade-matrix.csv   ←  seed matrix consumed by 5-2
```

## Method note (for the paper)

> *We project future bilateral aggregate trade using Stone's (1962) RAS method. The seed matrix is the 2018–2022 average bilateral trade in construction aggregates (sand, gravel/crushed stone, limestone). Future surplus and deficit vectors derived from our extraction–reserve and demand projections are imposed via iterative proportional fitting, preserving the structural pattern of historical trade while letting net trade volumes evolve.*

Reference: **Stone, R. (1962). Multiple classifications in social accounting.** *Bulletin of the International Statistical Institute*, **39**(3), 215–233.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
ROOT    = Path.cwd().parent
DATA    = ROOT / 'data'
RESULTS = ROOT / 'results'

# Bilateral matrix archive lives on F:/ (large files, not in the repo).
# Adjust this path if your archive is elsewhere.
BILATERAL_DIR = Path('F:/aggregates-mining-biodiversity/results/trade-data/trade-142')

YEAR_WINDOW = list(range(2018, 2023))   # 2018-2022 inclusive
CATS        = ['other-sand', 'gravel-crushed-stone', 'limestone']
assert BILATERAL_DIR.exists(), f'BILATERAL_DIR not found: {BILATERAL_DIR}'

## Load and average bilateral matrices

Sum the three materials, average over the 5-year window, convert kg → Mt. Result: a single 142 × 142 matrix representing the average annual bilateral aggregate trade pattern of the late 2010s / early 2020s.

In [3]:
annual_matrices = []
for y in YEAR_WINDOW:
    parts = []
    for cat in CATS:
        path = BILATERAL_DIR / f'{cat}-matrix-{y}.csv'
        if not path.exists():
            print(f'  missing: {path.name}')
            continue
        parts.append(pd.read_csv(path, index_col=0))
    if not parts:
        continue
    annual_matrices.append(sum(parts))   # sum across categories within the year
    print(f'  loaded {y}: {sum(p.shape[0] for p in parts) // len(parts)} countries, '
          f'{sum(parts).sum().sum() / 1e9:.1f} Mt total')

# Average across years; convert kg -> Mt for consistency with 5-2 units.
T_avg_Mt = sum(annual_matrices) / len(annual_matrices) / 1e9
print(f'\nAveraged matrix: {T_avg_Mt.shape}, sum = {T_avg_Mt.sum().sum():.2f} Mt/yr')

  loaded 2018: 142 countries, 334.7 Mt total
  loaded 2019: 142 countries, 327.7 Mt total
  loaded 2020: 142 countries, 313.5 Mt total
  loaded 2021: 142 countries, 330.7 Mt total
  loaded 2022: 142 countries, 286.7 Mt total

Averaged matrix: (142, 142), sum = 318.67 Mt/yr


## Sanity check: top historical corridors

These should match the well-known real-world aggregate trade routes: Gulf hub (UAE/Oman → Kuwait, Qatar, India), East Asian short-distance (China → Hong Kong; Indonesia/Malaysia → Singapore), Norwegian crushed-stone exports to Northern Europe, Canada → USA, Mexico → USA.

In [4]:
flows = T_avg_Mt.stack().sort_values(ascending=False)
print('=== Top 20 bilateral flows (Mt/yr, 2018–2022 avg) ===')
for (exp, imp), v in flows.head(20).items():
    if v > 0:
        print(f'  {exp} -> {imp}: {v:7.2f}')

=== Top 20 bilateral flows (Mt/yr, 2018–2022 avg) ===
  CHN -> HKG:   30.33
  MYS -> SGP:   24.43
  ARE -> IND:   22.03
  ARE -> KWT:   21.57
  OMN -> QAT:   14.56
  SUN -> SUN:   12.24
  CAN -> USA:   10.79
  NLD -> BEL:    8.67
  IDN -> SGP:    8.64
  DEU -> NLD:    8.56
  MEX -> USA:    8.17
  BEL -> FRA:    7.50
  NOR -> DEU:    5.33
  OMN -> ARE:    5.18
  NOR -> DNK:    4.67
  ARE -> BHR:    4.60
  ARE -> EWP:    4.19
  IND -> EWP:    4.02
  DEU -> BEL:    3.96
  NOR -> NLD:    3.87


In [5]:
# Marginal totals — should match the country-level UN data we used earlier.
marg = pd.DataFrame({
    'export_Mt_yr': T_avg_Mt.sum(axis=1),
    'import_Mt_yr': T_avg_Mt.sum(axis=0),
})
marg['net_import_Mt_yr'] = marg['import_Mt_yr'] - marg['export_Mt_yr']

print('=== Top 12 net exporters ===')
print(marg.nsmallest(12, 'net_import_Mt_yr').round(2))
print('\n=== Top 12 net importers ===')
print(marg.nlargest(12, 'net_import_Mt_yr').round(2))

=== Top 12 net exporters ===
     export_Mt_yr  import_Mt_yr  net_import_Mt_yr
ARE         56.03          5.47            -50.56
CHN         34.58          2.83            -31.75
MYS         27.52          0.23            -27.29
NOR         22.34          0.07            -22.28
OMN         22.49          0.48            -22.01
CAN         11.03          1.68             -9.35
MEX          8.19          0.27             -7.92
IDN          8.68          1.28             -7.40
GBR          6.70          3.11             -3.59
VNM          4.27          0.98             -3.29
DEU         18.49         15.84             -2.65
JPN          3.32          0.70             -2.62

=== Top 12 net importers ===
     export_Mt_yr  import_Mt_yr  net_import_Mt_yr
SGP          0.01         36.87             36.86
HKG          0.41         31.68             31.27
KWT          0.00         21.71             21.71
IND          4.25         25.91             21.67
USA          2.17         22.42          

## Add a tiny ε floor for novel pairs

Strict RAS / IPF on a sparse seed matrix cannot route any flow through pairs that historically traded zero. To preserve flexibility — e.g. a future scenario where a new corridor opens — we add a uniform floor `ε` equal to ~1/10000 of the mean nonzero flow. Structural pairs still dominate (the floor is ~4 orders of magnitude smaller), but IPF can use any country pair if it has to.

In [6]:
vals = T_avg_Mt.values
eps = float(vals[vals > 0].mean()) * 1e-4
T_seed = T_avg_Mt.copy()
T_seed.values[:] = vals + eps
np.fill_diagonal(T_seed.values, 0.0)   # no self-trade

n_pairs_original = (T_avg_Mt.values > 0).sum()
n_pairs_with_floor = (T_seed.values > 0).sum()
print(f'Floor ε = {eps:.4e} Mt/yr  (mean nonzero × 1e-4)')
print(f'Nonzero pairs before floor: {n_pairs_original:,}  ({100*n_pairs_original/(T_seed.shape[0]**2):.1f}% of matrix)')
print(f'Nonzero pairs after  floor: {n_pairs_with_floor:,}  ({100*n_pairs_with_floor/(T_seed.shape[0]**2):.1f}% of matrix)')
print(f'Floor contribution to total: {eps * (T_seed.shape[0]**2 - T_seed.shape[0]) / T_seed.sum().sum() * 100:.3f}%')

Floor ε = 8.4260e-06 Mt/yr  (mean nonzero × 1e-4)
Nonzero pairs before floor: 3,782  (18.8% of matrix)
Nonzero pairs after  floor: 20,022  (99.3% of matrix)
Floor contribution to total: 0.056%


## Save

In [7]:
out = RESULTS / 'historical-trade-matrix.csv'
T_seed.to_csv(out)
print(f'Saved: {out}')
print(f'  shape: {T_seed.shape}')
print(f'  total: {T_seed.sum().sum():.2f} Mt/yr')

Saved: d:\wty\Documents\GitHub\aggregate-mining-biodiversity\results\historical-trade-matrix.csv
  shape: (142, 142)
  total: 301.89 Mt/yr
